# Logistic Regression and Classification

In our previous notebook, we explored **Linear Regression** to predict continuous values like taxi fares. The output of a linear regression model can be any real number (from negative infinity to positive infinity).

But what if we want to predict a category instead of a continuous number? This is called **Classification**.
* **Binary Classification**: Predicting one of two possible categories (e.g., "Will it rain tomorrow?" -> Yes or No; "Is this email spam?" -> Spam or Not Spam).
* We typically represent the categories as `1` (Yes/Spam) and `0` (No/Not Spam).

### Why Linear Regression Fails for Classification
If we use our linear model equation $y = wx + b$ for classification, the model will draw a straight line.
1. The line can predict values like `1.5` or `-0.5`. What does a prediction of `1.5` mean when the only valid options are `0` or `1`? It makes no sense!
2. We need a model that outputs a **probability** (a value strictly between `0` and `1`) so we can interpret it as "there is an 80% chance of rain tomorrow".

### Entering Logistic Regression
This is where **Logistic Regression** comes in. Despite its name, Logistic Regression is actually a **classification** algorithm, not a regression algorithm.

It helps us by predicting the **probability** that a given input belongs to a specific category (e.g., a 75% probability that it will rain).

To do this, it takes our linear regression equation $z = wx + b$ (which can output any value) and maps it into a probability between 0 and 1 using the **Sigmoid Function**.

### Entering the Sigmoid Function
To solve this, we take the output of our linear equation (let's call it $z$):
$$z = w_1x_1 + w_2x_2 + ... + b$$

And we pass it through a mathematical function called the **Sigmoid Function** (also known as the Logistic Function), represented by $\sigma(z)$:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This function has an S-shaped curve that perfectly "squishes" any input value $z$ into a value between `0` and `1`:
* **Extreme Positive Input ($z \to \infty$)**: $e^{-z}$ becomes almost $0$, so $\sigma(z) \approx \frac{1}{1 + 0} = 1$.
* **Extreme Negative Input ($z \to -\infty$)**: $e^{-z}$ becomes extremely large, making the denominator huge. So $\sigma(z) \approx \frac{1}{\infty} = 0$.
* **Zero Input ($z = 0$)**: $e^0 = 1$, so $\sigma(0) = \frac{1}{1 + 1} = 0.5$ (exactly a 50% chance).


In [1]:
# @title Coding Example: How Sigmoid Helps Us Categorize
import numpy as np

# Let's define the Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Problem: Will a student pass the exam based on hours studied?
# Features (x): Hours studied
# Let's assume we already trained a model and found these parameters:
weight = 0.8  # weight (w)
bias = -3.0   # bias (b)

# Let's test for different study hours
study_hours = np.array([1, 3, 4, 6, 8])

# Step 1: Calculate the linear output (z)
z_values = (weight * study_hours) + bias

# Step 2: Pass through Sigmoid to get probabilities (y_predicted)
probabilities = sigmoid(z_values)

# Step 3: Categorize (Thresholding at 0.5: >= 0.5 is Pass (1), else Fail (0))
predictions = (probabilities >= 0.5).astype(int)

# Print the results clearly
print(f"{'Hours Studied':<15} | {'Linear Output (z)':<18} | {'Probability (y\')':<18} | {'Final Prediction':<15}")
print("-" * 75)
for h, z, prob, pred in zip(study_hours, z_values, probabilities, predictions):
    status = "Pass (1)" if pred == 1 else "Fail (0)"
    print(f"{h:<15} | {z:<18.2f} | {prob:<18.4f} | {status:<15}")


Hours Studied   | Linear Output (z)  | Probability (y')   | Final Prediction
---------------------------------------------------------------------------
1               | -2.20              | 0.0998             | Fail (0)       
3               | -0.60              | 0.3543             | Fail (0)       
4               | 0.20               | 0.5498             | Pass (1)       
6               | 1.80               | 0.8581             | Pass (1)       
8               | 3.40               | 0.9677             | Pass (1)       


## 📉 2. Key Concepts: Log Loss vs. Mean Squared Error (MSE)

When training our model, we need a way to calculate **how wrong** the model's guesses are. We call this **Loss**.

In **Linear Regression**, we used **Mean Squared Error (MSE)**, which calculates the average squared difference between the actual taxi fare and the predicted fare. 

However, for **Logistic Regression**, we use **Log Loss** instead. Here is why MSE doesn't work well here in a simple way:

### 1. MSE makes learning too slow when the model is wildly wrong:
*   Because Logistic Regression uses the Sigmoid function to force outputs between `0` and `1`, the curve becomes extremely flat at the very ends (near `0` and `1`).
*   If the model is highly confident but completely wrong (e.g., it predicts a `99%` chance of rain, but it turns out to be a sunny day), MSE's mathematical curve becomes so flat that the model barely learns from its mistake. It gets "stuck" and updates its weights at a snail's pace.
*   **Log Loss** fixes this by using a logarithmic scale. If the model is highly confident but wrong, Log Loss shoots up to infinity, giving the model a massive "nudge" to quickly correct itself.

### 2. MSE can create "traps" during training:
*   When we combine the Sigmoid function with MSE, the mathematical surface becomes wavy with many "valleys" (local traps). During training, the model can fall into one of these shallow valleys and think it has found the best solution, even though a much better valley exists further ahead.
*   **Log Loss** is mathematically designed to create a single, smooth, bowl-shaped valley. No matter where the model starts, it will always slide down smoothly to the very bottom (the best possible solution) without getting trapped.


## 🛡️ 3. Regularization and the "Runaway Weights" Problem

Overfitting happens when a model memorizes details of the training data instead of learning general patterns. **Regularization** helps prevent overfitting by penalizing large weights.

*   **L2 Regularization (Ridge)**: Adds the sum of the squares of the weights to the loss function. It shrinks weights close to zero (default in Scikit-Learn).
*   **L1 Regularization (Lasso)**: Adds the sum of the absolute values of weights. It can push some weights to exactly zero, which is useful for feature selection.

### Why Regularization is More Important in Logistic Regression than Linear Regression

In Linear Regression, weights generally stabilize naturally. But in Logistic Regression, we face a unique threat: **Perfect Linear Separability**.

Imagine we have a feature that perfectly separates the data:
*   Every student who studied $\ge 4$ hours passed ($y = 1$).
*   Every student who studied $< 4$ hours failed ($y = 0$).

The Logistic Regression model wants to fit this perfectly by predicting exactly $y' = 1$ for the pass cases and exactly $y' = 0$ for the fail cases. 
*   To get an output of exactly $1$ or $0$ from the Sigmoid function ($\sigma(z) = \frac{1}{1 + e^{-z}}$), the linear output $z = wx + b$ must be positive or negative **infinity**.
*   To make $z$ infinity, the weight $w$ must grow to **infinity**!

Without regularization, the model will continuously increase the weights to infinity during training, causing **runaway weights**. Regularization acts as a leash, penalizing large weights and keeping them finite.


In [ ]:
# @title Coding Example: Demonstrating Runaway Weights on Separable Data
from sklearn.linear_model import LogisticRegression
import numpy as np

# Perfectly linearly separable dataset
# Hours studied: 1, 2, 3 (fail, y=0) vs 5, 6, 7 (pass, y=1)
X = np.array([1, 2, 3, 5, 6, 7]).reshape(-1, 1)
y = np.array([0, 0, 0, 1, 1, 1])

# We will train Scikit-Learn LogisticRegression models with different regularization levels.
# C is the inverse of regularization strength: C = 1 / alpha.
# Smaller C means stronger regularization (tighter leash on weights).
# Massive C (like 1e9) means almost NO regularization (runaway weights).

for C_val in [0.1, 1.0, 1000.0, 1e9]:
    # We set solver='lbfgs', penalty is L2 by default
    model = LogisticRegression(C=C_val, solver='lbfgs')
    model.fit(X, y)
    print(f"Regularization Strength (C) = {C_val:<10} | Learned Weight (w) = {model.coef_[0][0]:.4f} | Bias (b) = {model.intercept_[0]:.4f}")


## 🔄 4. Training Loss, Validation Loss, and Early Stopping

When we train a machine learning model, we want it to perform well on **new, unseen data**, not just memorize the examples we show it. To check this, we split our data into two main sets:

### 📚 The Exam Analogy

Imagine a student preparing for a history exam:
1.  **Training Loss (Practice Questions)**: 
    *   These are the practice questions the student studies over and over again. 
    *   **Training Loss** measures the errors on these study questions. As the student spends more time studying, their score on these questions will naturally get better and better (Training Loss goes down).
2.  **Validation Loss (Mock Test)**: 
    *   This is a fresh mock test with brand-new questions the student has never seen before. 
    *   **Validation Loss** measures the errors on this mock test. This tells us if the student actually *understands* history, or if they have just *memorized* the answers to the practice questions.

---

### 📈 Overfitting and Early Stopping

*   **Early Phase**: At the start of studying, both Training Loss and Validation Loss go down. The student is learning general rules.
*   **Overfitting Phase**: Eventually, if the student studies *too* much, they will start memorizing the exact punctuation and typos of the practice questions. Their score on the practice questions will be perfect (Training Loss continues to drop to near $0$), but their score on the mock test will get **worse** (Validation Loss starts to rise). This is called **Overfitting**.

### How Early Stopping Helps
**Early Stopping** is a smart safety switch. During training, we monitor the Validation Loss after each study round (epoch). The moment the Validation Loss stops decreasing and begins to rise, we **stop training immediately**.

This ensures we keep the model at its peak performance on new data, before it has a chance to overfit and memorize the training set.


In [ ]:
# @title Coding Example: Simulating Gradient Descent with Early Stopping
import matplotlib.pyplot as plt
import numpy as np

# Let's simulate training and validation loss values over 50 epochs
epochs = np.arange(1, 51)

# Training loss continuously decreases
train_loss = 1.5 * np.exp(-epochs / 10) + 0.05

# Validation loss decreases at first, then starts increasing (overfitting)
val_loss = 1.5 * np.exp(-epochs / 8) + 0.02 * (epochs / 10) ** 2 + 0.08

# Let's find the exact epoch where validation loss is at its minimum
best_epoch = np.argmin(val_loss) + 1
best_loss = np.min(val_loss)

# Plotting the training progress
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss, label="Training Loss", color="blue", lw=2)
plt.plot(epochs, val_loss, label="Validation Loss", color="red", lw=2)
plt.axvline(x=best_epoch, color="green", linestyle="--", label=f"Early Stopping Trigger (Epoch {best_epoch})")
plt.scatter(best_epoch, best_loss, color="green", zorder=5)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Early Stopping: Stopping Training when Validation Loss Rises")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Training stopped early at epoch {best_epoch} to prevent overfitting!")
print(f"Lowest Validation Loss achieved: {best_loss:.4f}")


### Libraries Used in this Project

*   **Pandas:** Think of this as "Excel for Python." It helps us organize and clean data using tables called DataFrames.
*   **NumPy:** Used for fast mathematical calculations and handling large lists of numbers (arrays).
*   **Scikit-Learn (sklearn):** The primary toolbox for Machine Learning. It contains the algorithms and tools needed to train and test models.
    *   `train_test_split`: Helps us hide some data from the model so we can test it later.
    *   `LogisticRegression`: The "brain" that learns the patterns in the weather data.
    *   `accuracy_score`: A simple way to see what percentage of our predictions were correct.

### What are we doing in this exercise?

In this exercise, we are performing **Binary Classification**.

**The Goal:** We want to look at today's weather patterns (like Temperature, Humidity, and Pressure) and predict a "Yes" or "No" answer to the question: **"Will it rain tomorrow?"**

We use **Logistic Regression** because it is a great algorithm for predicting categories (like Yes/No). It looks at historical data, finds correlations between weather factors and rainfall, and creates a mathematical boundary to classify new weather data.

## Australian Rainfall Prediction

**Objective:** Build a classification model using Logistic Regression to predict whether it will rain tomorrow in Australia based on weather observations.

## Step 1: Download the Dataset

The dataset is sourced from Kaggle ('Rain in Australia'). Since we are running in **Google Colab**, we download it directly from our GitHub repository using a URL — no manual upload needed!

> **Note:** The full dataset (~145,000 rows) is available on [Kaggle](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package). If you have a Kaggle account, you can upload your `kaggle.json` API key to Colab and use the Kaggle API to download it directly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -------------------------------------------------------------------
# Load data directly from GitHub (works in Colab without any upload!)
# The dataset is the 'Rain in Australia' dataset from Kaggle.
# -------------------------------------------------------------------
DATA_URL = 'https://raw.githubusercontent.com/akshaykhanna/learning-complete-ai/main/weather.csv'

df = pd.read_csv(DATA_URL)

# Print shape and first 5 rows
print("Data Shape:", df.shape)
print(df.head())

Data Shape: (145460, 24)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity3pm  Pressure9am  \
0           W           44.0          W  ...        22.0       1007.7   
1         WNW           44.0        NNW  ...        25.0       1010.6   
2         WSW           46.0          W  ...        30.0       1007.6   
3          NE           24.0         SE  ...        16.0       1017.6   
4           W           41.0        ENE  ...        33.0       1010.8   

   Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  RainTomo

# Data Preprocessing

In this step, we need to clean the dataset by dropping redundant columns that do not contribute to the prediction or may cause data leakage. We also need to remove any null values to ensure the model can be trained effectively.

In [ ]:
# Drop redundant columns
df = df.drop(['Date', 'Location', 'RISK_MM'], axis=1)

# Remove null values
df = df.dropna()

print("Cleaned Data Shape:", df.shape)

Cleaned Data Shape: (56420, 21)


# Splitting Data & Building the Model

To build the model, we first isolate the target variable (`RainTomorrow`) from the features. We then perform a 75/25 train-test split to ensure we have a separate set of data to evaluate the model's performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Convert categorical variables to dummy/indicator variables
df = pd.get_dummies(df, drop_first=True)

# Isolate target variable
X = df.drop('RainTomorrow_Yes', axis=1)
y = df['RainTomorrow_Yes']

# Train-Test Split (75/25)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and fit the model
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=5000)

# Evaluation

After training the model, we test its accuracy on the unseen test data to see how well it generalizes to new observations.

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 85.66%
